In [ ]:
import os, glob, shutil, tempfile, time
import pandas as pd
import numpy as np
from collections import defaultdict
from IPython.display import HTML, display
from openpyxl import load_workbook

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
TARGET_WEEK = "2026_09_07"
SEND_EMAIL  = False

BASE_DIR   = os.path.join(os.path.expanduser("~"), "Concentrix Corporation", "WFM-Expedia-HCM - Branding files")
TA_FOLDER  = os.path.join(BASE_DIR, "Rawdata", "OUTPUT_TEAM_ALIGNMENT")
LEAVE_FILE = os.path.join(BASE_DIR, "Save", "Expedia_LeaveForm.xlsm")
EWS_FILE   = os.path.join(os.path.expanduser("~"), "Concentrix Corporation",
                          "WFM-Expedia-HCM - Branding files", "Headcount",
                          "HC Master Database - 2026.xlsx")
TEMP_DIR         = tempfile.mkdtemp()
EMAIL_TO = (
    'Abishek . <abishek.a@concentrix.com>; '
    'Rajat Roy <rajat.roy@concentrix.com>; '
    'Lokesh Yadav <lokesh.yadav2@concentrix.com>; '
    'Anirudh Chatterjee <anirudh.chatterjee1@concentrix.com>; '
    'Pankaj Shaw <pankaj.shaw@concentrix.com>; '
    'Sakshi . <sakshi.sakshi30@concentrix.com>'
)
EMAIL_CC = (
    'Varun Kathuria <varun.kathuria@concentrix.com>; '
    'Urmila Chakka <urmila.chakka1@concentrix.com>; '
    'Puneet Suneja <puneet.suneja@concentrix.com>; '
    'KIRPAN PATAR <kirpan.patar@concentrix.com>; '
    'Tran Tran <huynhductran.tran@concentrix.com>; '
    'Van Tran <van.tran@concentrix.com>'
)
LEAVE_WEEK_LABEL = f"Schedule_{TARGET_WEEK}"

_wp             = TARGET_WEEK.split("_")
WB_LABEL        = "WB" + _wp[0][2:] + _wp[1] + _wp[2]
WEEK_START_DATE = pd.Timestamp(f"{_wp[0]}-{_wp[1]}-{_wp[2]}")

LOB_DISPLAY = {
    "Lodging"    : "Lodging Chat",
    "Non_Lodging": "Non-Lodging Chat",
    "Support"    : "Other Task",
    "HPO"        : "Lodging Chat",
}
LOB_ROSTER_CYCLE = {
    "Lodging Chat": "Fourth week of cycle",
}

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def find_col(df, name):
    return next((c for c in df.columns if c.strip().lower() == name.strip().lower()), None)

def parse_to_date(val):
    if pd.isna(val) or str(val).strip() in ("", "nan", "None"):
        return None
    try:
        s = float(str(val).strip())
        if 40000 <= s <= 55000:
            return (pd.Timestamp("1899-12-30") + pd.Timedelta(days=s)).date()
    except:
        pass
    try:
        return pd.to_datetime(str(val)).date()
    except:
        return None

def load_safe(path, sheet_name):
    tmp_path = os.path.join(TEMP_DIR, f"_tmp_{os.path.basename(path)}")
    shutil.copy2(path, tmp_path)
    xl = pd.ExcelFile(tmp_path, engine="openpyxl")
    names = xl.sheet_names
    xl.close()
    print(f"  Sheets in {os.path.basename(path)}: {names}")
    actual = next((s for s in names if s.strip().lower() == sheet_name.strip().lower()), None)
    if actual is None:
        print(f"  Sheet '{sheet_name}' not found")
        return pd.DataFrame()
    df = pd.read_excel(tmp_path, sheet_name=actual, dtype=str, engine="openpyxl")
    for _ in range(5):
        try: os.remove(tmp_path); break
        except PermissionError: time.sleep(0.3)
    return df

def apply_cell_formats(path, sheet_name, date_cols=(), int_cols=()):
    wb = load_workbook(path)
    ws = wb[sheet_name]
    hdr = {cell.value: cell.column for cell in ws[1] if cell.value}
    for col_name in date_cols:
        if col_name in hdr:
            for r in range(2, ws.max_row + 1):
                ws.cell(r, hdr[col_name]).number_format = "YYYY-MM-DD"
    for col_name in int_cols:
        if col_name in hdr:
            for r in range(2, ws.max_row + 1):
                ws.cell(r, hdr[col_name]).number_format = "0"
    wb.save(path)
    wb.close()

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1: Team Alignment CSVs — FIX: remove quoting=3
# ══════════════════════════════════════════════════════════════════════════════
csv_files = glob.glob(os.path.join(TA_FOLDER, "Team_Alignment_*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in: {TA_FOLDER}")

dfs = []
for fpath in csv_files:
    fname = os.path.basename(fpath)
    try:
        # DEFAULT quoting — handles "Advisor I, Customer Service" as single field
        # quoting=3 was the root cause: split on comma inside quotes → all cols shifted
        tmp = pd.read_csv(fpath, encoding="utf-8-sig", dtype=str)
        tmp["Week"] = fname.replace("Team_Alignment_", "").replace(".csv", "")
        dfs.append(tmp)
    except Exception as e:
        print(f"Skipped {fname}: {e}")

df_all = pd.concat(dfs, ignore_index=True)

# Show first CSV columns for verification
print(f"CSV columns: {dfs[0].columns.tolist() if dfs else 'no files'}")
print(f"IEX ID sample : {df_all['IEX ID'].dropna().head(3).tolist() if 'IEX ID' in df_all.columns else 'N/A'}")
print(f"OracleID sample: {df_all['OracleID'].dropna().head(3).tolist() if 'OracleID' in df_all.columns else 'N/A'}")

for col in ["IEX ID", "OracleID", "TL ID"]:
    if col in df_all.columns:
        df_all[col] = pd.to_numeric(df_all[col], errors="coerce")
if "Date Start Week" in df_all.columns:
    df_all["Date Start Week"] = pd.to_datetime(df_all["Date Start Week"], errors="coerce").dt.date
for col in df_all.select_dtypes("object").columns:
    df_all[col] = df_all[col].replace({"nan": np.nan, "None": np.nan, "": np.nan})

print(f"Total rows: {len(df_all)} | Weeks: {sorted(df_all['Week'].dropna().unique())}")

df_week = df_all[df_all["Week"] == TARGET_WEEK].copy().reset_index(drop=True)
df_week = df_week[df_week["LOB"].fillna("").str.strip() != "HPO"].reset_index(drop=True)
print(f"Rows for {TARGET_WEEK}: {len(df_week)}")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2: File 1 — Team Alignment (Expedia VN - Team Alignment.xlsx)
# ══════════════════════════════════════════════════════════════════════════════
xlsx1_cols = [c for c in ["IEX ID","OracleID","Employee Name","Email Id","TL ID",
                           "Supervisor Name","Designation","LOB","LOB_Combine","Site",
                           "HC_File_Role","Date Start Week","Week"] if c in df_week.columns]
file1_path = os.path.join(TEMP_DIR, f"Expedia VN - Team Alignment.xlsx")
with pd.ExcelWriter(file1_path, engine="openpyxl", date_format="YYYY-MM-DD") as writer:
    df_week[xlsx1_cols].to_excel(writer, index=False, sheet_name="team_alignment")
print(f"File 1 saved: {os.path.basename(file1_path)}")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3: File 2 — Vietnam_Inputs (Planned Leave + Preference ShortTerm)
# ══════════════════════════════════════════════════════════════════════════════
print("\nLoading Leave log...")
df_leave = load_safe(LEAVE_FILE, "log")
print(f"  Rows: {len(df_leave)} | Cols: {df_leave.columns.tolist()}")

df_lf          = df_leave.copy()
rta_col        = find_col(df_lf, "RTA")
week_col       = find_col(df_lf, "Week")
leave_type_col = find_col(df_lf, "Leave Type")
date_col       = find_col(df_lf, "Date Leave")
iex_col        = find_col(df_lf, "IEX ID")

if date_col:
    df_lf[date_col] = df_lf[date_col].apply(parse_to_date)
if rta_col:        df_lf = df_lf[df_lf[rta_col].str.strip() == "Approve"]
if week_col:       df_lf = df_lf[df_lf[week_col].str.strip() == LEAVE_WEEK_LABEL]
if leave_type_col: df_lf = df_lf[df_lf[leave_type_col].str.strip() != "WO"]
print(f"  Planned Leave rows after filter: {len(df_lf)}")

if iex_col:
    df_lf[iex_col] = pd.to_numeric(df_lf[iex_col], errors="coerce")

WANT_1     = ["Date Leave", "IEX ID", "Employee Name", "Manager Name", "LOB", "Leave Type"]
cols_1     = [find_col(df_lf, w) for w in WANT_1 if find_col(df_lf, w)]
df_planned = df_lf[cols_1].reset_index(drop=True)

print("\nLoading Special Request...")
df_special = load_safe(LEAVE_FILE, "Special Request")
print(f"  Rows: {len(df_special)} | Cols: {df_special.columns.tolist()}")

file2_path = os.path.join(TEMP_DIR, f"Vietnam_Inputs_{WB_LABEL}.xlsx")
with pd.ExcelWriter(file2_path, engine="openpyxl", date_format="YYYY-MM-DD") as writer:
    df_planned.to_excel(writer, index=False, sheet_name="Planned Leave")
    df_special.to_excel(writer, index=False, sheet_name="Preference ShortTerm")

apply_cell_formats(file2_path, "Planned Leave",
                   date_cols=("Date Leave",), int_cols=("IEX ID",))
print(f"File 2 saved: {os.path.basename(file2_path)}")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4: Pivot Table
# ══════════════════════════════════════════════════════════════════════════════
lob_piv   = "LOB_Combine" if "LOB_Combine" in df_week.columns else "LOB"
count_col = "OracleID" if "OracleID" in df_week.columns else df_week.columns[0]

pivot = df_week.pivot_table(index="HC_File_Role", columns=lob_piv,
                             values=count_col, aggfunc="count", fill_value=0)
pivot.columns.name = None; pivot.index.name = None
pivot["Grand Total"] = pivot.sum(axis=1)
pivot = pd.concat([pivot, pivot.sum().rename("Grand Total").to_frame().T])
pivot = pivot[[c for c in pivot.columns if c != "Grand Total"] + ["Grand Total"]]

# ══════════════════════════════════════════════════════════════════════════════
# STEP 5: EWS Table
# ══════════════════════════════════════════════════════════════════════════════
print("\nLoading EWS...")
df_ews = load_safe(EWS_FILE, "EWS")
print(f"  Rows: {len(df_ews)} | Cols: {df_ews.columns.tolist()}")

ews_iex = find_col(df_ews, "IEX ID")
ews_nam = find_col(df_ews, "Employee Name")
ews_lob = find_col(df_ews, "LOB")
ews_sup = find_col(df_ews, "Supervisor Name")
ews_lwd = find_col(df_ews, "LWD Expected")

if ews_lwd:
    df_ews[ews_lwd] = pd.to_datetime(df_ews[ews_lwd], errors="coerce")
    df_ews_f = df_ews[df_ews[ews_lwd] >= WEEK_START_DATE].copy()
    df_ews_f[ews_lwd] = df_ews_f[ews_lwd].dt.date
else:
    df_ews_f = df_ews.copy()

if ews_iex:
    df_ews_f[ews_iex] = pd.to_numeric(df_ews_f[ews_iex], errors="coerce").astype("Int64")

if ews_lob:
    df_ews_f = df_ews_f[df_ews_f[ews_lob].fillna("").str.strip() != "HPO"]
ews_keep   = [c for c in [ews_iex, ews_nam, ews_lob, ews_sup, ews_lwd] if c]
df_ews_out = df_ews_f[ews_keep].reset_index(drop=True)
print(f"  EWS rows (LWD >= {WEEK_START_DATE.date()}): {len(df_ews_out)}")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 6: HTML Tables
# ══════════════════════════════════════════════════════════════════════════════
FONT = "font-family:Calibri,Arial,sans-serif;font-size:13px;"

def build_pivot_html(pivot_df, wb_label):
    D = "#1F3864"; M = "#2E4694"; E = "#EBF5FB"; O = "#FFFFFF"; T = "#AED6F1"
    TH = f"padding:7px 12px;border:1px solid #fff;color:#fff;text-align:center;{FONT}font-weight:bold;"
    cols = list(pivot_df.columns)
    h = (f'<table cellspacing="0" cellpadding="0" style="border-collapse:collapse;{FONT}margin:0 0 8px;">'
         f'<thead><tr><th colspan="{len(cols)+1}" style="{TH}background:{D};font-size:14px;'
         f'text-align:left;padding:10px 14px;">Team Alignment HC Summary &mdash; {wb_label}</th></tr>'
         f'<tr><th style="{TH}background:{M};text-align:left;">HC_File_Role</th>')
    for c in cols:
        h += f'<th style="{TH}background:{D if c=="Grand Total" else M};">{c}</th>'
    h += "</tr></thead><tbody>"
    for i, (role, row) in enumerate(pivot_df.iterrows()):
        tot = str(role) == "Grand Total"
        bg  = T if tot else (E if i % 2 == 0 else O)
        fw  = "bold" if tot else "normal"
        td  = f"padding:5px 10px;border:1px solid #D5D8DC;background:{bg};{FONT}font-weight:{fw};"
        h  += f'<tr><td style="{td}text-align:left;">{role}</td>'
        for c in cols:
            val = "" if (row[c] == 0 and not tot) else int(row[c])
            h  += f'<td style="{td}text-align:center;">{val}</td>'
        h += "</tr>"
    return h + "</tbody></table>"

def build_ews_html(df, week_start_str):
    H = "#C75000"; E_BG = "#FEF9F5"; O = "#FFFFFF"
    TH = f"padding:7px 12px;border:1px solid #fff;color:#fff;text-align:center;{FONT}font-weight:bold;"
    if df.empty:
        return (f'<table cellspacing="0" cellpadding="0" style="border-collapse:collapse;{FONT}margin:0 0 8px;">'
                f'<thead><tr><th style="{TH}background:{H};text-align:left;padding:10px 14px;">'
                f'EWS Cases &mdash; From {week_start_str} &mdash; No records found</th></tr></thead></table>')
    cols = list(df.columns)
    h = (f'<table cellspacing="0" cellpadding="0" style="border-collapse:collapse;{FONT}margin:0 0 8px;">'
         f'<thead><tr><th colspan="{len(cols)}" style="{TH}background:{H};font-size:14px;'
         f'text-align:left;padding:10px 14px;">EWS Cases &mdash; From {week_start_str}</th></tr><tr>')
    for c in cols:
        h += f'<th style="{TH}background:{H};">{c}</th>'
    h += "</tr></thead><tbody>"
    for i, (_, row) in enumerate(df.iterrows()):
        bg = E_BG if i % 2 == 0 else O
        td = f"padding:5px 10px;border:1px solid #F0CCBB;background:{bg};{FONT}"
        h += "<tr>"
        for c in cols:
            val = str(row[c]) if pd.notna(row[c]) and str(row[c]) not in ("", "nan", "<NA>") else ""
            h += f'<td style="{td}text-align:center;">{val}</td>'
        h += "</tr>"
    return h + "</tbody></table>"

# ══════════════════════════════════════════════════════════════════════════════
# STEP 7: Email Body
# ══════════════════════════════════════════════════════════════════════════════
grand_row = pivot.loc["Grand Total"]
total_hc  = int(grand_row["Grand Total"])

lob_grouped = defaultdict(int)
for lob_name in [c for c in pivot.columns if c != "Grand Total"]:
    lob_grouped[LOB_DISPLAY.get(lob_name, lob_name)] += int(grand_row[lob_name])

hc_items    = "\n".join(f"<li>{n}: {v} heads.</li>" for n, v in lob_grouped.items())
cycle_items = "\n".join(f"<li>{n}: {c}.</li>" for n, c in LOB_ROSTER_CYCLE.items())

pivot_html = build_pivot_html(pivot, WB_LABEL)
ews_html   = build_ews_html(df_ews_out, WEEK_START_DATE.strftime("%Y-%m-%d"))
FONT_S     = f"{FONT}color:#222;"

email_body = f"""<div style="padding:20px 24px;background:#ffffff;{FONT_S}">
  <p>Dear Team,</p><br>
  <p>I would like to update the input for <strong>{WB_LABEL}</strong> as below:</p>
  <ol>
    <li style="margin-bottom:8px;">
      Team Alignment, Leave and Shift Reference: as attached files: &#x201C;Vietnam_Input&#x2026;&#x201D; &amp; &#x201C;Team Alignment&#x201D;<br>
      Team alignment will be based on the TLs from Workday, no Mini Team.
    </li>
    <li style="margin-bottom:8px;">Roster Cycle:<ol>{cycle_items}</ol></li>
    <li style="margin-bottom:8px;">HC Active: total <strong>{total_hc}</strong> heads<ol>{hc_items}</ol></li>
  </ol>
  <br>{pivot_html}<br>{ews_html}<br>
  <p style="margin-top:20px;line-height:1.6;">
    Thanks &amp; Regards,<br>
    <strong>Chinh Nguyen</strong><br>
    Analyst, WFM Real Time Management<br>
    Level 4, Tower 1, OneHub Saigon, Lot C1-2, D1 Street, Saigon Hi Tech Park,<br>
    Tan Phu Ward, District 9, Ho Chi Minh City, Vietnam<br>
    Ph No: +84 986 473 419 | Email: <a href="mailto:huuchinh.nguyen@concentrix.com">huuchinh.nguyen@concentrix.com</a>
  </p>
</div>"""

WB_SHORT      = "WB" + _wp[1] + _wp[2]
EMAIL_SUBJECT = f"Expedia - Scheduling inputs: {WB_SHORT}"

display(HTML(email_body))
print(f"\nFile 1 : {os.path.basename(file1_path)}")
print(f"File 2 : {os.path.basename(file2_path)}  (Planned Leave | Preference ShortTerm)")
print(f"EWS    : {len(df_ews_out)} rows")

# ══════════════════════════════════════════════════════════════════════════════
# SEND EMAIL
# ══════════════════════════════════════════════════════════════════════════════
if SEND_EMAIL:
    import subprocess, pythoncom, psutil, win32com.client

    def send_email_outlook(to, cc, subject, body, attachments=None, quit_after=False):
        pythoncom.CoInitialize()
        was_on = any(p.name().lower() == "outlook.exe" for p in psutil.process_iter(["name"]))
        if not was_on:
            for exe in [r"C:\Program Files\Microsoft Office\root\Office16\OUTLOOK.EXE",
                        r"C:\Program Files (x86)\Microsoft Office\root\Office16\OUTLOOK.EXE"]:
                if os.path.exists(exe): subprocess.Popen([exe]); break
            for _ in range(30):
                time.sleep(1)
                try: win32com.client.GetActiveObject("Outlook.Application"); break
                except: pass
        try:
            ol = win32com.client.Dispatch("Outlook.Application")
            ol.GetNamespace("MAPI").Logon()
            mail = ol.CreateItem(0)
            mail.To = to; mail.CC = cc
            mail.Subject = subject; mail.HTMLBody = body
            for att in (attachments or []):
                if os.path.exists(os.path.abspath(att)):
                    mail.Attachments.Add(os.path.abspath(att))
                    print(f"Attached: {os.path.basename(att)}")
            mail.Send()
            print(f"Email sent to: {to}")
            time.sleep(3)
        finally:
            if quit_after and not was_on:
                try: ol.Quit()
                except: pass

    send_email_outlook(EMAIL_TO, EMAIL_CC, EMAIL_SUBJECT, email_body, [file1_path, file2_path])

CSV columns: ['IEX ID', 'OracleID', 'Employee Name', 'Email Id', 'TL ID', 'Supervisor Name', 'Designation', 'LOB', 'LOB_Combine', 'Site', 'HC_File_Role', 'Date Start Week', 'Week']
IEX ID sample : ['3109399', '3052315', '3093306']
OracleID sample: ['103293766', '103060467', '103139871']
Total rows: 5074 | Weeks: ['2026_01_05', '2026_01_12', '2026_01_19', '2026_01_26', '2026_02_02', '2026_02_09', '2026_02_16', '2026_02_23', '2026_03_02', '2026_03_09', '2026_03_16', '2026_03_23', '2026_03_30', '2026_04_06', '2026_04_13', '2026_04_20', '2026_04_27', '2026_05_04', '2026_05_11', '2026_05_18', '2026_05_25', '2026_06_01', '2026_06_08', '2026_06_15', '2026_06_22', '2026_06_29', '2026_07_06', '2026_07_13', '2026_07_20', '2026_07_27', '2026_08_03', '2026_08_10', '2026_08_17', '2026_08_24', '2026_08_31', '2026_09_07', '2026_09_14', '2026_09_21', '2026_09_28', '2026_10_05', '2026_10_12', '2026_10_19', '2026_10_26', '2026_11_02']
Rows for 2026_08_31: 111
File 1 saved: Expedia VN - Team Alignment.xl


File 1 : Expedia VN - Team Alignment.xlsx
File 2 : Vietnam_Inputs_WB260831.xlsx  (Planned Leave | Preference ShortTerm)
EWS    : 5 rows
